In [1]:
# import custom_scorer_v4
# import spacy
# nlp = spacy.load("model-best")
# from spacy.training import Corpus
# from spacy.training.example import Example
# test_corpus = Corpus("valid_FAERS_R1_v1_SME1_new_revision.spacy")
# test_data = list(test_corpus(nlp))
# scores = nlp.evaluate(test_data)
# scores
import custom_scorer_v5
import spacy
nlp = spacy.load("../model/SMESMEoutput_FAERS_test_data_v1/model-best")
from spacy.training import Corpus
from spacy.training.example import Example
test_corpus = Corpus("../data/processed_data/valid_FAVERS_R1_v1_SME1_new_revision.spacy")
#test_corpus = Corpus("valid.spacy")
test_data = list(test_corpus(nlp))

/compute001/lwu/projects/LLM4AE/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
len(test_data)

2155

In [ ]:
# import csv
# with open('Results/ner_scores_FAERS_R1_Testing.csv', mode='w', newline='') as f:
#     writer = csv.writer(f)
#     writer.writerow(["Type", "Precision", "Recall", "F1"])
#     for ent_type, metrics in scores["ents_per_type"].items():
#         writer.writerow([
#             ent_type,
#             metrics.get("precision", ""),
#             metrics.get("recall", ""),
#             metrics.get("f1", "")
#         ])
#     writer.writerow([
#         "TOTAL",
#         scores["ents_p"],
#         scores["ents_r"],
#         scores["ents_f"]
#     ])

In [2]:
def eval_pred(gold, pred, case_id=None):
    M, C, S, N = 0, 0, 0, 0  # Initialize counts
    label_counts = defaultdict(lambda: {"M": 0, "C": 0, "S": 0, "N": 0})
    if True:
        # Convert entity labels to lowercase for case-insensitive comparison
        gold_ents = list(gold)
        pred_ents = list(pred)
        pred_flag = [False] * len(pred_ents)
        sorted_pred_ents = sorted(pred_ents, key=lambda x: (int(x[0]), int(x[1])))
        sorted_gold_ents = sorted(gold_ents, key=lambda x: (int(x[0]), int(x[1])))
        M_temp = 0
        C_temp = 0
        # Calculate the number of exact matches, partial matches, false positives, and false negatives
        for gold_ent in sorted_gold_ents:
            totally_match_found = False
            partial_match_found = False
            for ind1 in range(len(sorted_pred_ents)):
                pred_ent = sorted_pred_ents[ind1]
                if pred_ent[0] == gold_ent[0] and pred_ent[1] == gold_ent[1] and pred_ent[2] == gold_ent[2]:
                    totally_match_found = True
                    pred_flag[ind1] = True
                    M += 1
                    M_temp += 1
                    label_counts[gold_ent[2]]["M"] += 1
                    break
                elif pred_ent[0] == gold_ent[0] or pred_ent[1] == gold_ent[1] or gold_ent[0] < pred_ent[0] < gold_ent[1] or gold_ent[0] < pred_ent[1] < gold_ent[1] or (pred_ent[0] < gold_ent[0] and pred_ent[1] > gold_ent[0]):
                    if not pred_flag[ind1]:
                        partial_match_found = True
                        pred_flag[ind1] = True
                        C += 1
                        C_temp += 1
                        label_counts[gold_ent[2]]["C"] += 1
                        break
            if not (totally_match_found or partial_match_found):
                N += 1
                label_counts[gold_ent[2]]["N"] += 1
        S += len(sorted_pred_ents) - M_temp - C_temp
        for ind1 in range(len(sorted_pred_ents)):
            if not pred_flag[ind1]:
                label_counts[sorted_pred_ents[ind1][2]]["S"] += 1
    # Revised M', C', S', N'
    M_ = M + (0.5 * C)
    C_ = 0.5 * C
    S_ = 0.25 * S
    N_ = N
    # Compute revised Precision / Recall / F1
    precision = M_ / (M_ + C_ + S_) if (M_ + C_ + S_) > 0 else 0
    recall = M_ / (M_ + C_ + N_) if (M_ + C_ + N_) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    # Compute metrics for each label type
    label_metrics = []
    for label, counts in label_counts.items():
        M_l = counts["M"]
        C_l = counts["C"]
        S_l = counts["S"]
        N_l = counts["N"]
        M_l_ = M_l + (0.5 * C_l)
        C_l_ = 0.5 * C_l
        S_l_ = 0.25 * S_l
        N_l_ = N_l
        precision_l = M_l_ / (M_l_ + C_l_ + S_l_) if (M_l_ + C_l_ + S_l_) > 0 else 0
        recall_l = M_l_ / (M_l_ + C_l_ + N_l_) if (M_l_ + C_l_ + N_l_) > 0 else 0
        f1_l = (2 * precision_l * recall_l) / (precision_l + recall_l) if (precision_l + recall_l) > 0 else 0
        label_metrics.append({
            'case_id':case_id,
            'label':label,
            "precision": precision_l,
            "recall": recall_l,
            "f1": f1_l,
            "Matched":M_l,
            "Partially Matched":C_l,
            "Missed":N_l,
            "FP":S_l
        })
    return label_metrics


In [ ]:
#what I added
from collections import defaultdict
from typing import List, Tuple, Dict, Any
import pandas as pd
import spacy

# ---------- 句子切分器（只用于找整句） ----------
_nlp_sent = spacy.blank("en")
_nlp_sent.add_pipe("sentencizer")

def _sentences(text: str):
    """返回[(start, end, sent_text), ...]"""
    doc = _nlp_sent(text)
    return [(s.start_char, s.end_char, s.text) for s in doc.sents]


# ---------- 辅助：区间重叠长度 ----------
def _overlap_len(a0, a1, b0, b1):
    return max(0, min(a1, b1) - max(a0, b0))


# ---------- 主函数：对齐单个样本，产出对齐行 ----------
def align_case_rows(case_id: int, text: str,
                    gold_ents: List[Tuple[int,int,str]],
                    pred_ents: List[Tuple[int,int,str]]) -> List[Dict[str, Any]]:
    """
    返回一组行，每行是一对 gold/pred 的对齐结果：
      - match_type: 'M' 完全匹配, 'C' 部分匹配, 'N' 只存在 gold(漏检), 'S' 只存在 pred(假阳)
      - 同时保留 gold/pred 的 start/end/text、标签，以及整句文本
    """
    rows = []
    # 句子边界（用于取整句）
    sents = _sentences(text)
    # 为了取实体原文，准备一个快速函数
    def etext(start, end):
        return text[start:end]
    # 先做可匹配的 pred 标记
    pred_flag = [False] * len(pred_ents)
    # 排序（与原逻辑一致）
    gold_sorted = sorted(gold_ents, key=lambda x: (int(x[0]), int(x[1])))
    pred_sorted = sorted([(p[0], p[1], p[2], idx) for idx, p in enumerate(pred_ents)],
                         key=lambda x: (int(x[0]), int(x[1])))
    # gold 驱动匹配：优先完全匹配，其次选择重叠最多的部分匹配
    for g0, g1, glab in gold_sorted:
        # 找包含该 gold 的句子
        sent_text = ""
        for ss, ee, stxt in sents:
            if g0 >= ss and g1 <= ee:
                sent_text = stxt
                break
        exact_idx = None
        best_partial_idx = None
        best_partial_ov = 0
        for p0, p1, plab, orig_idx in pred_sorted:
            if pred_flag[orig_idx]:
                continue
            # 完全匹配：边界与标签都相同
            if (p0 == g0) and (p1 == g1) and (plab == glab):
                exact_idx = (p0, p1, plab, orig_idx)
                break
            # 部分匹配：任一端点相同或区间有重叠
            if (p0 == g0) or (p1 == g1) or (g0 < p0 < g1) or (g0 < p1 < g1) or (p0 < g0 < p1):
                ov = _overlap_len(g0, g1, p0, p1)
                if ov > best_partial_ov:
                    best_partial_ov = ov
                    best_partial_idx = (p0, p1, plab, orig_idx)
        if exact_idx is not None:
            p0, p1, plab, orig_i = exact_idx
            pred_flag[orig_i] = True
            rows.append({
                "case_id": case_id,
                "match_type": "M",
                "label_gold": glab,
                "gold_start": g0, "gold_end": g1, "gold_text": etext(g0, g1),
                "label_pred": plab,
                "pred_start": p0, "pred_end": p1, "pred_text": etext(p0, p1),
                "sentence": sent_text
            })
        elif best_partial_idx is not None:
            p0, p1, plab, orig_i = best_partial_idx
            pred_flag[orig_i] = True
            rows.append({
                "case_id": case_id,
                "match_type": "C",
                "label_gold": glab,
                "gold_start": g0, "gold_end": g1, "gold_text": etext(g0, g1),
                "label_pred": plab,
                "pred_start": p0, "pred_end": p1, "pred_text": etext(p0, p1),
                "sentence": sent_text
            })
        else:
            # 漏检 N（没有匹配的 pred）
            rows.append({
                "case_id": case_id,
                "match_type": "N",
                "label_gold": glab,
                "gold_start": g0, "gold_end": g1, "gold_text": etext(g0, g1),
                "label_pred": None,
                "pred_start": None, "pred_end": None, "pred_text": None,
                "sentence": sent_text
            })
    # 剩余未使用的 pred 作为 FP (S)
    for (p0, p1, plab, orig_i) in pred_sorted:
        if pred_flag[orig_i]:
            continue
        # 找该 pred 所在句子
        sent_text = ""
        for ss, ee, stxt in sents:
            if p0 >= ss and p1 <= ee:
                sent_text = stxt
                break
        rows.append({
            "case_id": case_id,
            "match_type": "S",
            "label_gold": None,
            "gold_start": None, "gold_end": None, "gold_text": None,
            "label_pred": plab,
            "pred_start": p0, "pred_end": p1, "pred_text": etext(p0, p1),
            "sentence": sent_text
        })
    return rows

# ---------- 批量：从 test_data + nlp 生成总表 ----------
def build_alignment_table(test_data, nlp) -> pd.DataFrame:
    """
    对 test_data 逐条预测并与 gold 对齐，返回一个包含：
      case_id, match_type(M/C/N/S), gold/pred 的 start/end/text/label, sentence
    的对齐明细表
    """
    all_rows = []
    for i, example in enumerate(test_data, 1):
        text = example.text
        # gold
        gold_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
        # pred
        pred_doc = nlp(text)
        pred_ents = [(ent.start_char, ent.end_char, ent.label_) for ent in pred_doc.ents]
        rows = align_case_rows(i, text, gold_ents, pred_ents)
        all_rows.extend(rows)
    df = pd.DataFrame(all_rows, columns=[
        "case_id", "match_type",
        "label_gold", "gold_start", "gold_end", "gold_text",
        "label_pred", "pred_start", "pred_end", "pred_text",
        "sentence"
    ])
    return df


# 假设你已有：
# test_data = list(Corpus("your_valid.spacy")(nlp))
# nlp = spacy.load("model-best")

df_align = build_alignment_table(test_data, nlp)

# 看前几行
print(df_align.head())

# 如果只看某个 case_id：
df_align[df_align["case_id"] == 42]


import pandas as pd

# 显示所有列
pd.set_option('display.max_columns', None)

# 如果想恢复默认
# pd.reset_option('display.max_columns')

print(df_align.head())   # 现在就会显示全部列，没有省略号



NameError: name 'test_data' is not defined

In [5]:
#What I added
import spacy
import pandas as pd
from collections import defaultdict
import difflib

# ==== 句子切分器（只用于拿整句文本和句内相对偏移）====
_nlp_sent = spacy.blank("en")
_nlp_sent.add_pipe("sentencizer")

def _doc_sents(doc):
    return [(s.start_char, s.end_char, s.text) for s in _nlp_sent(doc.text).sents]


# ==== 句子对齐：先 exact，必要时可开 fuzzy ====
def align_sentences_both(docs_pred, docs_gold, fuzzy=False, fuzzy_threshold=0.92):
    """
    返回：
      pairs: [(p_doc_i, p_sent_i, p_sent_info), (g_doc_j, g_sent_j, g_sent_info)] 的列表
      unmatched_gold: [(g_doc_j, g_sent_j, g_sent_info)] gold 侧未对齐句
      unmatched_pred: [(p_doc_i, p_sent_i, p_sent_info)] pred 侧未对齐句
    """
    # 预处理两侧句子
    pred_index = defaultdict(list)  # text -> [(di, si, (start,end,text))]
    pred_all = []  # [(di, si, (start,end,text))]
    for di, dp in enumerate(docs_pred):
        sents = _doc_sents(dp)
        for si, s in enumerate(sents):
            pred_index[s[2]].append((di, si, s))
            pred_all.append((di, si, s))
    gold_all = []  # [(gj, sj, (start,end,text))]
    for gj, dg in enumerate(docs_gold):
        sents = _doc_sents(dg)
        for sj, s in enumerate(sents):
            gold_all.append((gj, sj, s))
    used_pred = set()
    used_gold = set()
    pairs = []
    # 1) exact
    for gj, sj, gs in gold_all:
        if (gj, sj) in used_gold:
            continue
        candidates = pred_index.get(gs[2], [])
        picked = None
        for (di, si, ps) in candidates:
            if (di, si) in used_pred:
                continue
            picked = (di, si, ps)
            break
        if picked is not None:
            pairs.append(((picked[0], picked[1], picked[2]), (gj, sj, gs)))
            used_pred.add((picked[0], picked[1]))
            used_gold.add((gj, sj))
    # 2) fuzzy（可选）
    if fuzzy:
        for gj, sj, gs in gold_all:
            if (gj, sj) in used_gold:
                continue
            best = None
            best_score = 0.0
            for di, si, ps in pred_all:
                if (di, si) in used_pred:
                    continue
                score = difflib.SequenceMatcher(None, ps[2], gs[2]).ratio()
                if score > best_score:
                    best = (di, si, ps)
                    best_score = score
            if best and best_score >= fuzzy_threshold:
                pairs.append(((best[0], best[1], best[2]), (gj, sj, gs)))
                used_pred.add((best[0], best[1]))
                used_gold.add((gj, sj))
    # 未对齐的 gold / pred 句子
    unmatched_gold = []
    for gj, sj, gs in gold_all:
        if (gj, sj) not in used_gold:
            unmatched_gold.append((gj, sj, gs))
    unmatched_pred = []
    for di, si, ps in pred_all:
        if (di, si) not in used_pred:
            unmatched_pred.append((di, si, ps))
    return pairs, unmatched_gold, unmatched_pred

# ==== 把句内实体转为相对偏移（便于在不同 doc 上对比）====
def ents_in_sentence_relative(doc, sent_start, sent_end):
    triples = []
    for ent in doc.ents:
        if ent.start_char >= sent_start and ent.end_char <= sent_end:
            triples.append((ent.start_char - sent_start,
                            ent.end_char - sent_start,
                            ent.label_))
    return triples


# ==== 匹配工具 ====
def _overlap_len(a0, a1, b0, b1):
    return max(0, min(a1, b1) - max(a0, b0))


def match_gold_pred_in_sentence(text, gold_tris, pred_tris):
    """
    在一个句子里对齐 gold/pred 实体，返回多行记录：
      match_type ∈ {'M','C','N','S'}
      并带上 gold/pred 的 start/end/text/label
    """
    rows = []
    pred_flag = [False]*len(pred_tris)
    # 先 gold 驱动：优先 exact，再选重叠最长的 partial
    for g0, g1, glab in sorted(gold_tris, key=lambda x:(x[0], x[1])):
        exact_idx = None
        best_partial_idx = None
        best_ov = 0
        for j, (p0, p1, plab) in enumerate(sorted(pred_tris, key=lambda x:(x[0], x[1]))):
            if pred_flag[j]:
                continue
            if (p0 == g0) and (p1 == g1) and (plab == glab):
                exact_idx = j
                break
            # 部分匹配判定（与你的C一致）
            if (p0 == g0) or (p1 == g1) or (g0 < p0 < g1) or (g0 < p1 < g1) or (p0 < g0 < p1):
                ov = _overlap_len(g0, g1, p0, p1)
                if ov > best_ov:
                    best_ov = ov
                    best_partial_idx = j
        if exact_idx is not None:
            p0, p1, plab = pred_tris[exact_idx]
            pred_flag[exact_idx] = True
            rows.append({
                "match_type": "M",
                "label_gold": glab, "gold_start": g0, "gold_end": g1, "gold_text": text[g0:g1],
                "label_pred": plab, "pred_start": p0, "pred_end": p1, "pred_text": text[p0:p1],
            })
        elif best_partial_idx is not None:
            p0, p1, plab = pred_tris[best_partial_idx]
            pred_flag[best_partial_idx] = True
            rows.append({
                "match_type": "C",
                "label_gold": glab, "gold_start": g0, "gold_end": g1, "gold_text": text[g0:g1],
                "label_pred": plab, "pred_start": p0, "pred_end": p1, "pred_text": text[p0:p1],
            })
        else:
            rows.append({
                "match_type": "N",
                "label_gold": glab, "gold_start": g0, "gold_end": g1, "gold_text": text[g0:g1],
                "label_pred": None, "pred_start": None, "pred_end": None, "pred_text": None,
            })
    # 剩余未用到的 pred → S
    for j, (p0, p1, plab) in enumerate(pred_tris):
        if pred_flag[j]:
            continue
        rows.append({
            "match_type": "S",
            "label_gold": None, "gold_start": None, "gold_end": None, "gold_text": None,
            "label_pred": plab, "pred_start": p0, "pred_end": p1, "pred_text": text[p0:p1],
        })
    return rows


# ==== 构建 df_align：把 LLM(docs1) & SME(docs2) 对齐到明细表 ====
def build_df_align_from_spacy(docs_llm, docs_sme, fuzzy=False, fuzzy_threshold=0.92):
    """
    输出 DataFrame 列包含：
      doc_id_pred, sent_id_pred, doc_id_gold, sent_id_gold, sentence_gold, sentence_pred,
      match_type, gold/pred start/end/text/label
    """
    pairs, unmatched_gold, unmatched_pred = align_sentences_both(
        docs_pred=docs_llm, docs_gold=docs_sme, fuzzy=fuzzy, fuzzy_threshold=fuzzy_threshold
    )
    all_rows = []
    # 已对齐的句子：句内相对偏移对齐
    for (di, si, (ps0, ps1, ptxt)), (gj, sj, (gs0, gs1, gtxt)) in pairs:
        # gold 用 gold 句文本、pred 用 pred 句文本，各自句内偏移
        gold_tris = ents_in_sentence_relative(docs_sme[gj], gs0, gs1)
        pred_tris = ents_in_sentence_relative(docs_llm[di], ps0, ps1)
        rows = match_gold_pred_in_sentence(gtxt, gold_tris, pred_tris)  # 用 gold 句文本取片段
        for r in rows:
            r.update({
                "doc_id_pred": di, "sent_id_pred": si, "sentence_pred": ptxt,
                "doc_id_gold": gj, "sent_id_gold": sj, "sentence_gold": gtxt,
            })
        all_rows.extend(rows)
    # gold 未对齐句：全算 N
    for (gj, sj, (gs0, gs1, gtxt)) in unmatched_gold:
        gold_tris = ents_in_sentence_relative(docs_sme[gj], gs0, gs1)
        for (g0, g1, glab) in gold_tris:
            all_rows.append({
                "doc_id_pred": None, "sent_id_pred": None, "sentence_pred": None,
                "doc_id_gold": gj, "sent_id_gold": sj, "sentence_gold": gtxt,
                "match_type": "N",
                "label_gold": glab, "gold_start": g0, "gold_end": g1, "gold_text": gtxt[g0:g1],
                "label_pred": None, "pred_start": None, "pred_end": None, "pred_text": None,
            })
    # pred 未对齐句：全算 S
    for (di, si, (ps0, ps1, ptxt)) in unmatched_pred:
        pred_tris = ents_in_sentence_relative(docs_llm[di], ps0, ps1)
        for (p0, p1, plab) in pred_tris:
            all_rows.append({
                "doc_id_pred": di, "sent_id_pred": si, "sentence_pred": ptxt,
                "doc_id_gold": None, "sent_id_gold": None, "sentence_gold": None,
                "match_type": "S",
                "label_gold": None, "gold_start": None, "gold_end": None, "gold_text": None,
                "label_pred": plab, "pred_start": p0, "pred_end": p1, "pred_text": ptxt[p0:p1],
            })
    df_align = pd.DataFrame(all_rows, columns=[
        "doc_id_pred", "sent_id_pred", "doc_id_gold", "sent_id_gold",
        "sentence_gold", "sentence_pred",
        "match_type",
        "label_gold", "gold_start", "gold_end", "gold_text",
        "label_pred", "pred_start", "pred_end", "pred_text",
    ])
    return df_align


def load_spacy_file(file_path):
    """Load a .spacy file and return a list of Doc objects"""
    nlp = spacy.blank("en")
    doc_bin = DocBin().from_disk(file_path)
    return list(doc_bin.get_docs(nlp.vocab))

from spacy.tokens import DocBin

# 载入两份 .spacy
docs_llm = load_spacy_file("valid_FAVERS_R1_v1_LLM_new_revision.spacy")   # 作为 pred
# docs_ether = load_spacy_file("valid_FAVERS_R1_v1_ETHER_new_revision.spacy")
docs_sme = load_spacy_file("valid_FAVERS_R1_v1_SME1_new_revision.spacy")  # 作为 gold

# 生成 df_align；先关掉 fuzzy，确保精确对齐更稳，再按需打开
df_align1 = build_df_align_from_spacy(docs_llm, docs_sme, fuzzy=False)

# df_align2 = build_df_align_from_spacy(docs_ether, docs_sme, fuzzy=False)

# 看前几行（显示全部列）
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
print(df_align1.head(20))

# 只看漏检 N：
print(df_align1[df_align1["match_type"]=="N"].head(20))

# 保存
# df_align.to_csv("df_align_llm_vs_sme.csv", index=False)


FileNotFoundError: [Errno 2] No such file or directory: 'valid_FAVERS_R1_v1_LLM_new_revision.spacy'

: 

In [ ]:
import spacy
from spacy.tokens import DocBin

# 1) 准备 nlp（一定要有 sentencizer）
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

def load_spacy_file(path):
    db = DocBin().from_disk(path)
    return list(db.get_docs(nlp.vocab))  # 用同一个 nlp.vocab


# 2) 提取“以空格开头”的句子；若原 Doc 没有 sents，就用 nlp(doc.text) 临时分句
def extract_sentences_with_space(docs):
    out = []
    for doc in docs:
        if not doc.has_annotation("SENT_START"):
            doc_for_sents = nlp(doc.text)     # 仅用于拿句子边界
        else:
            doc_for_sents = doc
        for s in doc_for_sents.sents:
            if s.text.startswith(" "):        # 开头是空白
                out.append(s.text)
    return out


# 3) 加载两个文件
docs_llm = load_spacy_file("valid_FAVERS_R1_v1_LLM_new_revision.spacy")
docs_sme = load_spacy_file("valid_FAVERS_R1_v1_SME1_new_revision.spacy")

# 4) 统计并找交集
llm_sents = extract_sentences_with_space(docs_llm)
sme_sents = extract_sentences_with_space(docs_sme)

common = set(llm_sents) & set(sme_sents)

print(f"LLM 开头有空格的句子数: {len(llm_sents)}")
print(f"SME 开头有空格的句子数: {len(sme_sents)}")
print(f"两边完全相同（且开头有空格）的句子数: {len(common)}")
for s in list(common)[:10]:
    print(repr(s))


In [ ]:
#What I added
import spacy
import pandas as pd
from collections import defaultdict
import difflib

# ==== 句子切分器（只用于拿整句文本和句内相对偏移）====
_nlp_sent = spacy.blank("en")
_nlp_sent.add_pipe("sentencizer")

def _doc_sents(doc):
    return [(s.start_char, s.end_char, s.text) for s in _nlp_sent(doc.text).sents]

def normalize_entity_label(label: str) -> str:
    label = label.lower()
    if label in ['drug', 'cdrug', 'sdrug']:
        return 'drug'
    elif label in ['mhx', 'fhx', 'hx']:
        return 'hx'
    elif label in ['ae', 'mae']:
        return 'ae'
    elif label in ['ro', 'r/o']:
        return 'ro'
    elif label in ['second_level_diagnosis', 'diagnostic', 'symptom']:
        return 'ae'
    elif label in ['cause_of_death']:
        return 'cod'
    elif label in ['medical_history']:
        return 'medical history'
    else:
        return label


def ents_in_sentence_relative(doc, sent_start, sent_end, label_mapper=None):
    triples = []
    for ent in doc.ents:
        if ent.start_char >= sent_start and ent.end_char <= sent_end:
            lab = ent.label_
            if label_mapper is not None:
                lab = label_mapper(lab)
            triples.append((ent.start_char - sent_start,
                            ent.end_char - sent_start,
                            lab))
    return triples

# ==== 构建 df_align：允许对 pred/gold 侧分别指定映射 ====
def build_df_align_from_spacy(docs_llm, docs_sme, fuzzy=False, fuzzy_threshold=0.92,
                              mapper_pred=None, mapper_gold=None):
    """
    输出 DataFrame 列包含：
      doc_id_pred, sent_id_pred, doc_id_gold, sent_id_gold, sentence_gold, sentence_pred,
      match_type, gold/pred start/end/text/label
    """
    pairs, unmatched_gold, unmatched_pred = align_sentences_both(
        docs_pred=docs_llm, docs_gold=docs_sme, fuzzy=fuzzy, fuzzy_threshold=fuzzy_threshold
    )
    all_rows = []
    # 已对齐的句子：句内相对偏移对齐
    for (di, si, (ps0, ps1, ptxt)), (gj, sj, (gs0, gs1, gtxt)) in pairs:
        gold_tris = ents_in_sentence_relative(docs_sme[gj], gs0, gs1, label_mapper=mapper_gold)
        pred_tris = ents_in_sentence_relative(docs_llm[di], ps0, ps1, label_mapper=mapper_pred)
        rows = match_gold_pred_in_sentence(gtxt, gold_tris, pred_tris)  # 用 gold 句文本取片段
        for r in rows:
            r.update({
                "doc_id_pred": di, "sent_id_pred": si, "sentence_pred": ptxt,
                "doc_id_gold": gj, "sent_id_gold": sj, "sentence_gold": gtxt,
            })
        all_rows.extend(rows)
    # gold 未对齐句：全算 N
    for (gj, sj, (gs0, gs1, gtxt)) in unmatched_gold:
        gold_tris = ents_in_sentence_relative(docs_sme[gj], gs0, gs1, label_mapper=mapper_gold)
        for (g0, g1, glab) in gold_tris:
            all_rows.append({
                "doc_id_pred": None, "sent_id_pred": None, "sentence_pred": None,
                "doc_id_gold": gj, "sent_id_gold": sj, "sentence_gold": gtxt,
                "match_type": "N",
                "label_gold": glab, "gold_start": g0, "gold_end": g1, "gold_text": gtxt[g0:g1],
                "label_pred": None, "pred_start": None, "pred_end": None, "pred_text": None,
            })
    # pred 未对齐句：全算 S
    for (di, si, (ps0, ps1, ptxt)) in unmatched_pred:
        pred_tris = ents_in_sentence_relative(docs_llm[di], ps0, ps1, label_mapper=mapper_pred)
        for (p0, p1, plab) in pred_tris:
            all_rows.append({
                "doc_id_pred": di, "sent_id_pred": si, "sentence_pred": ptxt,
                "doc_id_gold": None, "sent_id_gold": None, "sentence_gold": None,
                "match_type": "S",
                "label_gold": None, "gold_start": None, "gold_end": None, "gold_text": None,
                "label_pred": plab, "pred_start": p0, "pred_end": p1, "pred_text": ptxt[p0:p1],
            })
    df_align = pd.DataFrame(all_rows, columns=[
        "doc_id_pred", "sent_id_pred", "doc_id_gold", "sent_id_gold",
        "sentence_gold", "sentence_pred",
        "match_type",
        "label_gold", "gold_start", "gold_end", "gold_text",
        "label_pred", "pred_start", "pred_end", "pred_text",
    ])
    return df_align



def load_spacy_file(file_path):
    """Load a .spacy file and return a list of Doc objects"""
    nlp = spacy.blank("en")
    doc_bin = DocBin().from_disk(file_path)
    return list(doc_bin.get_docs(nlp.vocab))

from spacy.tokens import DocBin

# 载入两份 .spacy
# docs_llm = load_spacy_file("valid_FAVERS_R1_v1_LLM_new_revision.spacy")   # 作为 pred
docs_ether = load_spacy_file("valid_FAVERS_R1_v1_ETHER_new_revision.spacy")
docs_sme = load_spacy_file("valid_FAVERS_R1_v1_SME1_new_revision.spacy")  # 作为 gold

# 生成 df_align；先关掉 fuzzy，确保精确对齐更稳，再按需打开
# df_align1 = build_df_align_from_spacy(docs_llm, docs_sme, fuzzy=False)

df_align2 = build_df_align_from_spacy(docs_ether, docs_sme, fuzzy=False, mapper_pred=normalize_entity_label, mapper_gold=normalize_entity_label)

# 看前几行（显示全部列）
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
print(df_align1.head(20))

# 只看漏检 N：
print(df_align1[df_align1["match_type"]=="N"].head(20))

# 保存
# df_align.to_csv("df_align_llm_vs_sme.csv", index=False)


In [ ]:
###if I save the predicted .spacy file and then compare, theres is no issues.
df_align_sorted = df_align[df_align["match_type"].isin(["M", "C", "N"])]
df_align1_sorted = df_align1[df_align1["match_type"].isin(["M", "C", "N"])]
# df_align1_sorted["sentence_gold"] = df_align1_sorted["sentence_gold"].str.lstrip()
df_align2_sorted = df_align2[df_align2["match_type"].isin(["M", "C", "N"])]
# df_align2_sorted["sentence_gold"] = df_align2_sorted["sentence_gold"].str.lstrip()

# df_align1_sorted["sentence"] = df_align1_sorted["sentence_gold"].fillna(df_align1_sorted["sentence_pred"])
# df_align2_sorted["sentence"] = df_align2_sorted["sentence_gold"].fillna(df_align2_sorted["sentence_pred"])
df_align1_sorted["sentence"] = df_align1_sorted["sentence_gold"]
df_align2_sorted["sentence"] = df_align2_sorted["sentence_gold"]
df_align_sorted["sentence"] = df_align_sorted["sentence_gold"]


# df_align1_sorted["sentence"] = df_align1_sorted["sentence"].str.replace(r"\s+", " ", regex=True).str.strip()
# df_align_sorted["sentence"]  = df_align_sorted["sentence"].str.replace(r"\s+", " ", regex=True).str.strip()
# df_align2_sorted["sentence"]  = df_align2_sorted["sentence"].str.replace(r"\s+", " ", regex=True).str.strip()


df_align_sorted = df_align_sorted.sort_values(by="sentence", kind="mergesort").reset_index(drop=True)
df_align1_sorted = df_align1_sorted.sort_values(by="sentence", kind="mergesort").reset_index(drop=True)
df_align2_sorted = df_align2_sorted.sort_values(by="sentence", kind="mergesort").reset_index(drop=True)


import numpy as np
min_len = min(len(df_align_sorted), len(df_align1_sorted))

# 找出不一样的位置
diff_idx = np.where(
    df_align_sorted["sentence"].iloc[:min_len].values != 
    df_align1_sorted["sentence"].iloc[:min_len].values
)[0]

print("两边 sentence 顺序不一样的行号:", diff_idx.tolist())


import numpy as np
min_len = min(len(df_align_sorted), len(df_align2_sorted))

# 找出不一样的位置
diff_idx = np.where(
    df_align_sorted["sentence"].iloc[:min_len].values != 
    df_align2_sorted["sentence"].iloc[:min_len].values
)[0]

print("两边 sentence 顺序不一样的行号:", diff_idx.tolist())

In [ ]:
db = DocBin()  # 用来存放预测的Doc
for eg in test_data:
    doc = eg.predicted  # 取预测结果
    db.add(doc)

# 保存为新的 .spacy 文件
db.to_disk("predicted_output.spacy")
print("预测结果已保存到 predicted_output.spacy")


In [ ]:
# #What I added
# import spacy
# import pandas as pd
# from collections import defaultdict
# import difflib

# # ==== 句子切分器（只用于拿整句文本和句内相对偏移）====
# _nlp_sent = spacy.blank("en")
# _nlp_sent.add_pipe("sentencizer")

# def _doc_sents(doc):
#     return [(s.start_char, s.end_char, s.text) for s in _nlp_sent(doc.text).sents]

# def normalize_entity_label(label: str) -> str:
#     label = label.lower()
#     if label in ['drug', 'cdrug', 'sdrug']:
#         return 'drug'
#     elif label in ['mhx', 'fhx', 'hx']:
#         return 'hx'
#     elif label in ['ae', 'mae']:
#         return 'ae'
#     elif label in ['ro', 'r/o']:
#         return 'ro'
#     elif label in ['second_level_diagnosis', 'diagnostic', 'symptom']:
#         return 'ae'
#     elif label in ['cause_of_death']:
#         return 'cod'
#     elif label in ['medical_history']:
#         return 'medical history'
#     else:
#         return label


# def ents_in_sentence_relative(doc, sent_start, sent_end, label_mapper=None):
#     triples = []
#     for ent in doc.ents:
#         if ent.start_char >= sent_start and ent.end_char <= sent_end:
#             lab = ent.label_
#             if label_mapper is not None:
#                 lab = label_mapper(lab)
#             triples.append((ent.start_char - sent_start,
#                             ent.end_char - sent_start,
#                             lab))
#     return triples

# # ==== 构建 df_align：允许对 pred/gold 侧分别指定映射 ====
# def build_df_align_from_spacy(docs_llm, docs_sme, fuzzy=False, fuzzy_threshold=0.92,
#                               mapper_pred=None, mapper_gold=None):
#     """
#     输出 DataFrame 列包含：
#       doc_id_pred, sent_id_pred, doc_id_gold, sent_id_gold, sentence_gold, sentence_pred,
#       match_type, gold/pred start/end/text/label
#     """
#     pairs, unmatched_gold, unmatched_pred = align_sentences_both(
#         docs_pred=docs_llm, docs_gold=docs_sme, fuzzy=fuzzy, fuzzy_threshold=fuzzy_threshold
#     )
#     all_rows = []
#     # 已对齐的句子：句内相对偏移对齐
#     for (di, si, (ps0, ps1, ptxt)), (gj, sj, (gs0, gs1, gtxt)) in pairs:
#         gold_tris = ents_in_sentence_relative(docs_sme[gj], gs0, gs1, label_mapper=mapper_gold)
#         pred_tris = ents_in_sentence_relative(docs_llm[di], ps0, ps1, label_mapper=mapper_pred)
#         rows = match_gold_pred_in_sentence(gtxt, gold_tris, pred_tris)  # 用 gold 句文本取片段
#         for r in rows:
#             r.update({
#                 "doc_id_pred": di, "sent_id_pred": si, "sentence_pred": ptxt,
#                 "doc_id_gold": gj, "sent_id_gold": sj, "sentence_gold": gtxt,
#             })
#         all_rows.extend(rows)
#     # gold 未对齐句：全算 N
#     for (gj, sj, (gs0, gs1, gtxt)) in unmatched_gold:
#         gold_tris = ents_in_sentence_relative(docs_sme[gj], gs0, gs1, label_mapper=mapper_gold)
#         for (g0, g1, glab) in gold_tris:
#             all_rows.append({
#                 "doc_id_pred": None, "sent_id_pred": None, "sentence_pred": None,
#                 "doc_id_gold": gj, "sent_id_gold": sj, "sentence_gold": gtxt,
#                 "match_type": "N",
#                 "label_gold": glab, "gold_start": g0, "gold_end": g1, "gold_text": gtxt[g0:g1],
#                 "label_pred": None, "pred_start": None, "pred_end": None, "pred_text": None,
#             })
#     # pred 未对齐句：全算 S
#     for (di, si, (ps0, ps1, ptxt)) in unmatched_pred:
#         pred_tris = ents_in_sentence_relative(docs_llm[di], ps0, ps1, label_mapper=mapper_pred)
#         for (p0, p1, plab) in pred_tris:
#             all_rows.append({
#                 "doc_id_pred": di, "sent_id_pred": si, "sentence_pred": ptxt,
#                 "doc_id_gold": None, "sent_id_gold": None, "sentence_gold": None,
#                 "match_type": "S",
#                 "label_gold": None, "gold_start": None, "gold_end": None, "gold_text": None,
#                 "label_pred": plab, "pred_start": p0, "pred_end": p1, "pred_text": ptxt[p0:p1],
#             })
#     df_align = pd.DataFrame(all_rows, columns=[
#         "doc_id_pred", "sent_id_pred", "doc_id_gold", "sent_id_gold",
#         "sentence_gold", "sentence_pred",
#         "match_type",
#         "label_gold", "gold_start", "gold_end", "gold_text",
#         "label_pred", "pred_start", "pred_end", "pred_text",
#     ])
#     return df_align



# def load_spacy_file(file_path):
#     """Load a .spacy file and return a list of Doc objects"""
#     nlp = spacy.blank("en")
#     doc_bin = DocBin().from_disk(file_path)
#     return list(doc_bin.get_docs(nlp.vocab))

# from spacy.tokens import DocBin

# # 载入两份 .spacy
# # docs_llm = load_spacy_file("valid_FAVERS_R1_v1_LLM_new_revision.spacy")   # 作为 pred
# docs_bert = load_spacy_file("predicted_output.spacy")
# docs_sme = load_spacy_file("valid_FAVERS_R1_v1_SME1_new_revision.spacy")  # 作为 gold

# # 生成 df_align；先关掉 fuzzy，确保精确对齐更稳，再按需打开
# # df_align1 = build_df_align_from_spacy(docs_llm, docs_sme, fuzzy=False)

# df_align = build_df_align_from_spacy(docs_bert, docs_sme, fuzzy=False, mapper_pred=normalize_entity_label, mapper_gold=normalize_entity_label)

# # 看前几行（显示全部列）
# import pandas as pd
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)
# print(df_align1.head(20))

# # 只看漏检 N：
# print(df_align1[df_align1["match_type"]=="N"].head(20))

# # 保存
# # df_align.to_csv("df_align_llm_vs_sme.csv", index=False)


In [ ]:
def compute_weighted_metrics(df_align: pd.DataFrame):
    """
    从 df_align 计算加权版 Precision / Recall / F1
    规则：
      M' = M + 0.5*C
      C' = 0.5*C
      S' = 0.25*S
      N' = N
    返回 overall 指标 + per-label DataFrame
    """
    # --- Overall ---
    cnt = df_align["match_type"].value_counts().to_dict()
    M = int(cnt.get("M", 0))
    C = int(cnt.get("C", 0))
    S = int(cnt.get("S", 0))
    N = int(cnt.get("N", 0))
    M_ = M + 0.5 * C
    C_ = 0.5 * C
    S_ = 0.25 * S
    N_ = N
    prec_overall = M_ / (M_ + C_ + S_) if (M_ + C_ + S_) > 0 else 0.0
    rec_overall  = M_ / (M_ + C_ + N_) if (M_ + C_ + N_) > 0 else 0.0
    f1_overall   = (2*prec_overall*rec_overall)/(prec_overall+rec_overall) if (prec_overall+rec_overall)>0 else 0.0
    overall = {
        "M": M, "C": C, "S": S, "N": N,
        "precision": prec_overall,
        "recall": rec_overall,
        "f1": f1_overall
    }
    # --- Per-label ---
    labels = pd.unique(pd.concat([
        df_align.loc[df_align["match_type"].isin(["M","C","N"]), "label_gold"].dropna(),
        df_align.loc[df_align["match_type"].eq("S"), "label_pred"].dropna()
    ]))
    rows = []
    for lab in labels:
        M_l = int(((df_align["match_type"]=="M") & (df_align["label_gold"]==lab)).sum())
        C_l = int(((df_align["match_type"]=="C") & (df_align["label_gold"]==lab)).sum())
        N_l = int(((df_align["match_type"]=="N") & (df_align["label_gold"]==lab)).sum())
        S_l = int(((df_align["match_type"]=="S") & (df_align["label_pred"]==lab)).sum())
        M_l_ = M_l + 0.5 * C_l
        C_l_ = 0.5 * C_l
        S_l_ = 0.25 * S_l
        N_l_ = N_l
        p_l = M_l_ / (M_l_ + C_l_ + S_l_) if (M_l_ + C_l_ + S_l_) > 0 else 0.0
        r_l = M_l_ / (M_l_ + C_l_ + N_l_) if (M_l_ + C_l_ + N_l_) > 0 else 0.0
        f1_l = (2*p_l*r_l)/(p_l+r_l) if (p_l+r_l)>0 else 0.0
        rows.append({
            "label": lab,
            "M": M_l, "C": C_l, "S": S_l, "N": N_l,
            "precision": p_l, "recall": r_l, "f1": f1_l
        })
    df_per_label = pd.DataFrame(rows).sort_values("label").reset_index(drop=True)
    return overall, df_per_label


def pretty_print_weighted(name, overall, df_per_label):
    print(f"\n===== {name} =====")
    print(f"Counts: M={overall['M']}  C={overall['C']}  S={overall['S']}  N={overall['N']}")
    print(f"Overall  P={overall['precision']:.4f}  R={overall['recall']:.4f}  F1={overall['f1']:.4f}")
    if not df_per_label.empty:
        print("\nPer-Label:")
        print(df_per_label.to_string(index=False))

overall1, df_lbl1 = compute_weighted_metrics(df_align1)
pretty_print_weighted("LLM vs SME (Weighted)", overall1, df_lbl1)

overall2, df_lbl2 = compute_weighted_metrics(df_align2)
pretty_print_weighted("ETHER vs SME (Weighted)", overall2, df_lbl2)


overall, df_lbl = compute_weighted_metrics(df_align)
pretty_print_weighted("ETHER vs SME (Weighted)", overall, df_lbl)

df_align["match_type"].value_counts()
# 保存 per-label 的指标
df_lbl1.to_csv("per_label_LLM_weighted.csv", index=False, encoding="utf-8-sig")
df_lbl2.to_csv("per_label_ETHER_weighted.csv", index=False, encoding="utf-8-sig")
df_lbl.to_csv("per_label_BERT_weighted.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([overall1]).to_csv("overall_LLM_weighted.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([overall2]).to_csv("overall_ETHER_weighted.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([overall]).to_csv("overall_BERT_weighted.csv", index=False, encoding="utf-8-sig")

In [ ]:
# import pandas as pd

# # 假设：
# # df_align1 = LLM 对齐结果（有 sentence_gold、label_gold 等）
# # df_align  = 原对齐结果（有 sentence、label_gold 等）
# # 1) 先统一句子列（用作匹配）
# df_align1["sentence"] = df_align1["sentence_gold"].fillna(df_align1["sentence_pred"])
# df_align["sentence"] = df_align["sentence"]
# # 2) 选择合并所需的列
# key_cols = ["sentence", "label_gold", "gold_start", "gold_end"]
# # 3) 给两边列加前缀，防止冲突
# df1_prefixed = df_align1.add_prefix("llm_")
# df2_prefixed = df_align.add_prefix("orig_")
# # 4) 把匹配键的列名还原成统一的（方便 merge）
# for col in key_cols:
#     df1_prefixed[col] = df_align1[col]
#     df2_prefixed[col] = df_align[col]


# # 5) 进行 merge（保留两边的所有数据）
# merged = pd.merge(
#     df1_prefixed,
#     df2_prefixed,
#     on=key_cols,
#     how="outer",
#     suffixes=("_llm", "_orig")
# )

# # 6) 查看结果
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', None)
# print(merged.head(10))

# merged['orig_match_type'].value_counts(dropna=False)


In [ ]:

# df_align1["sentence"] = df_align1["sentence_gold"].fillna(df_align1["sentence_pred"])
# df_align2["sentence"] = df_align2["sentence_gold"].fillna(df_align2["sentence_pred"])
# df_align["sentence"] = df_align["sentence"]
# # key_cols = ["sentence", "label_gold", "gold_start", "gold_end","gold_text","label_pred","pred_start","pred_end"]
# key_cols = ["sentence", "label_gold", "gold_start", "gold_end","gold_text","label_pred","pred_start","pred_end"]
# # 给每个表加前缀
# df1_prefixed = df_align1.add_prefix("llm_")
# df2_prefixed = df_align.add_prefix("bert_")
# df3_prefixed = df_align2.add_prefix("ether_")  # 举例叫 bert

# # 把 key 列名还原
# for col in key_cols:
#     df1_prefixed[col] = df_align1[col]
#     df2_prefixed[col] = df_align[col]
#     df3_prefixed[col] = df_align2[col]

# # 第一次 merge
# merged12 = pd.merge(df1_prefixed, df2_prefixed, on=key_cols, how="outer")

# # 再和第三个 merge
# merged_all = pd.merge(merged12, df3_prefixed, on=key_cols, how="outer")

# print(merged_all.shape)
# print(merged_all.head())
# merged_all['bert_match_type'].value_counts(dropna=False)
# df_m = merged_all[merged_all["bert_match_type"] == "M"]
# print(df_m)

# # 给每个表加前缀
# df1_prefixed = df_align1.add_prefix("llm_")
# df2_prefixed = df_align.add_prefix("bert_")
# df3_prefixed = df_align2.add_prefix("ether_")  # 举例叫 bert

# # 把 key 列名还原
# for col in key_cols:
#     df1_prefixed[col] = df_align1[col]
#     df2_prefixed[col] = df_align[col]
#     df3_prefixed[col] = df_align2[col]

# # 第一次 merge
# merged12 = pd.merge(df1_prefixed, df2_prefixed, on=key_cols, how="outer")

# # 再和第三个 merge
# key_cols1 = ["sentence", "label_gold", "gold_start", "gold_end","gold_text","llm_label_pred","llm_pred_start","llm_pred_end"]
# merged_all = pd.merge(merged12, df3_prefixed, on=key_cols1, how="outer")

# print(merged_all.shape)
# print(merged_all.head())
# merged_all['bert_match_type'].value_counts(dropna=False)
# df_m = merged_all[merged_all["bert_match_type"] == "M"]
# print(df_m)


In [ ]:
# # -*- coding: utf-8 -*-
# # 依赖：pandas
# import pandas as pd

# # ===================== 你需要提供的输入 =====================
# # f0, f1, f2: 三个 DataFrame
# # keys: 作为主键的列名列表，例如 keys = ["doc_id", "span_id"]
# # 示例（请用你自己的）：
# # keys = ["doc_id", "span_id"]

# # ===================== 参数：优先级与可选 tie-break 字段 =====================
# # match_type 优先级：M > C > N；未知统一最低
# PRIO = {"M": 0, "C": 1, "N": 2}

# # 可选的 tie-break 字段（按顺序越靠前优先级越高；不存在就自动跳过）
# # 比如你有 score、confidence、updated_at 等列，可加入此列表
# TIE_BREAK_COLS = [
#     # ("score", "desc"),         # 分数高者优先
#     # ("confidence", "desc"),    # 置信高者优先
#     # ("updated_at", "desc"),    # 时间新者优先
# ]

# # 是否统一把键列转成字符串、NFKC 规约、去多余空格
# NORMALIZE_KEYS = True

# # 是否把三表键集合对齐为“交集”
# ALIGN_TO_INTERSECTION = True

# # 若最终仍有重复，是否强制兜底：每侧保留首条（会丢弃多余行）
# FORCE_UNIQUE_FALLBACK = True


# # ===================== 工具函数 =====================
# def normalize_keys(df: pd.DataFrame, keys):
#     """统一键列的形态：转字符串、NFKC 规约、折叠空白、去首尾空白。"""
#     df = df.copy()
#     for k in keys:
#         s = df[k].astype(str)
#         # NFKC 规约（需要 pandas 2.x 内置的 str.normalize）
#         try:
#             s = s.str.normalize("NFKC")
#         except Exception:
#             # 某些旧环境可能无 normalize；则跳过这一行
#             pass
#         s = s.str.replace(r"\s+", " ", regex=True).str.strip()
#         df[k] = s
#     return df


# # def squash_to_unique(df: pd.DataFrame, keys, prio_map=PRIO, tie_break_cols=TIE_BREAK_COLS):
# #     """
# #     按业务规则把 DataFrame 压成“键唯一”：
# #     1) match_type 优先级（M<C<N<其他）
# #     2) pred_text 越长越优先
# #     3) 可选的 tie-break 列（desc/asc）
# #     4) 保留原始相对顺序作为最后的平手
# #     """
# #     df = df.copy()
# #     # 安全占位
# #     if "match_type" not in df.columns:
# #         df["match_type"] = pd.NA
# #     if "pred_text" not in df.columns:
# #         df["pred_text"] = pd.NA
# #     # 主排序：match_type 优先级
# #     df["_prio"] = df["match_type"].map(prio_map).fillna(9).astype(int)
# #     # 次排序：pred_text 越长越优先（负号实现降序）
# #     df["_plen_neg"] = -df["pred_text"].fillna("").astype(str).str.len()
# #     # 可选 tie-breaks
# #     # 为每个 tie-break 生成一个排序键；desc 用负号；asc 直接值
# #     extra_sort_cols = []
# #     for col, order in tie_break_cols:
# #         if col not in df.columns:
# #             continue
# #         # 用数值/时间优先；若为字符串，仍按 pandas 排序
# #         if order == "desc":
# #             # 尝试转换为数值；失败则原样
# #             z = pd.to_numeric(df[col], errors="ignore")
# #             # 对于可比较对象，desc 用“辅助负号”仅适用于数值；非数值直接加一个标识列
# #             if pd.api.types.is_numeric_dtype(z):
# #                 df[f"_tb_{col}_neg"] = -z
# #                 extra_sort_cols.append(f"_tb_{col}_neg")
# #             else:
# #                 # 非数值：无法用负号，直接用一个辅助列；通过 ascending=False 控制
# #                 # 这里简单起见，仍作为列加入排序列表，后面统一 ascending 列表
# #                 df[f"_tb_{col}"] = df[col]
# #                 extra_sort_cols.append((f"_tb_{col}", "desc"))
# #         else:
# #             df[f"_tb_{col}"] = df[col]
# #             extra_sort_cols.append((f"_tb_{col}", "asc"))
# #     # 最后：平手用原始行序
# #     df["_row"] = range(len(df))
# #     # 组装排序键与升降序
# #     sort_cols = list(keys) + ["_prio", "_plen_neg"] + [("_row", "asc")]
# #     # 把 extra_sort_cols 展开到 (name, dir) 形式
# #     expanded = []
# #     for c in extra_sort_cols:
# #         if isinstance(c, tuple):
# #             expanded.append(c)
# #         else:
# #             expanded.append((c, "asc"))
# #     sort_cols = sort_cols + expanded
# #     by = [c if isinstance(c, str) else c[0] for c in sort_cols]
# #     ascending = [True] * len(keys) + [True, True]  # keys升序, _prio升序, _plen_neg升序(实际是 pred 长度降序)
# #     ascending += [True]  # _row 升序
# #     # 为 extra_sort_cols 指定方向
# #     for c in expanded:
# #         ascending.append(False if c[1] == "desc" else True)
# #     # 稳定排序，保证分组后 first() 可控
# #     df = df.sort_values(by=by, ascending=ascending, kind="mergesort")
# #     out = df.groupby(keys, as_index=False, sort=False,dropna=False).first()
# #     # 清理临时列
# #     drop_cols = ["_prio", "_plen_neg", "_row"] + [x[0] if isinstance(x, tuple) else x for x in expanded]
# #     out = out.drop(columns=drop_cols, errors="ignore")
# #     return out
# def dedup_on_keys(df):
#     return df.drop_duplicates(subset=keys1, keep="first").copy()
#     # return df.copy()





# def dup_report(df: pd.DataFrame, name: str, keys, topn=10):
#     """报告 DataFrame 在 keys 上的重复情况，并展示若干重复样例。"""
#     g = df.groupby(keys, dropna=False).size().reset_index(name="_n")
#     bad = g[g["_n"] > 1].sort_values("_n", ascending=False)
#     if bad.empty:
#         print(f"— {name}：键唯一 ✅")
#         return bad
#     print(f"— {name}：发现 {len(bad)} 个非唯一键，示例（最多 {topn} 个键）：")
#     print(bad.head(topn))
#     # 展开部分重复键的具体行
#     ex_keys = [tuple(x) for x in bad.head(min(topn, 5))[keys].to_numpy()]
#     view = df[df[keys].apply(tuple, axis=1).isin(ex_keys)]
#     print(f"\n{name} 重复键明细（前若干行）：")
#     print(view.sort_values(keys).head(50))
#     return bad


# def with_prefix(df: pd.DataFrame, prefix: str, keys):
#     """为非键列加前缀，避免合并后重名冲突。"""
#     return df.rename(columns={c: f"{prefix}{c}" for c in df.columns if c not in keys})


# def force_unique_by_keys(df: pd.DataFrame, keys):
#     """兜底：按键去重保留首条（会静默丢弃重复）。"""
#     return df.sort_values(keys, kind="mergesort").drop_duplicates(subset=keys, keep="first")


# # ===================== 主流程 =====================
# def run_merge_pipeline(f0: pd.DataFrame, f1: pd.DataFrame, f2: pd.DataFrame, keys):
#     # 0) 键标准化（可选）
#     if NORMALIZE_KEYS:
#         f0 = normalize_keys(f0, keys)
#         f1 = normalize_keys(f1, keys)
#         f2 = normalize_keys(f2, keys)
#     # 1) 规则压扁为“键唯一”
#     f0u  = dedup_on_keys(df_align1)
#     f1u = dedup_on_keys(df_align)
#     f2u= dedup_on_keys(df_align2)
#     # f0u = squash_to_unique(f0, keys, prio_map=PRIO, tie_break_cols=TIE_BREAK_COLS)
#     # f1u = squash_to_unique(f1, keys, prio_map=PRIO, tie_break_cols=TIE_BREAK_COLS)
#     # f2u = squash_to_unique(f2, keys, prio_map=PRIO, tie_break_cols=TIE_BREAK_COLS)
#     # 2) 唯一性自检
#     print(">>> 压扁后唯一性检查")
#     b0 = dup_report(f0u, "BERT(压扁后)", keys)
#     b1 = dup_report(f1u, "LLM(压扁后)", keys)
#     b2 = dup_report(f2u, "ETHER(压扁后)", keys)
#     # 如果任何一侧仍不唯一，建议在 squash_to_unique 里增加更强的 tie-break
#     if not b0.empty or not b1.empty or not b2.empty:
#         print("\n⚠️ 仍存在非唯一键：请考虑在 TIE_BREAK_COLS 中增加更强的 tie-break 规则（例如 score/时间等）。")
#     # 3) 对齐三表共同键（可选）
#     if ALIGN_TO_INTERSECTION:
#         k0 = set(map(tuple, f0u[keys].itertuples(index=False, name=None)))
#         k1 = set(map(tuple, f1u[keys].itertuples(index=False, name=None)))
#         k2 = set(map(tuple, f2u[keys].itertuples(index=False, name=None)))
#         common = k0 & k1 & k2
#         if not (k0 == k1 == k2):
#             print("⚠️ 三表键集合不完全一致，使用交集对齐。")
#             f0u = f0u[f0u[keys].apply(tuple, axis=1).isin(common)]
#             f1u = f1u[f1u[keys].apply(tuple, axis=1).isin(common)]
#             f2u = f2u[f2u[keys].apply(tuple, axis=1).isin(common)]
#     # 4) 前缀并尝试严格 one_to_one 合并
#     llm_p  = with_prefix(f1u, "llm_",  keys)
#     bert_p = with_prefix(f0u, "bert_", keys)
#     eth_p  = with_prefix(f2u, "ether_",keys)
#     # 先探测 LLM×BERT
#     try:
#         _ = llm_p.merge(bert_p, on=keys, how="left", validate="one_to_one")
#         print("LLM×BERT ✅ one_to_one")
#         tmp = llm_p.merge(bert_p, on=keys, how="left", validate="one_to_one")
#     except Exception as e:
#         print("LLM×BERT ❌", e)
#         # 判定哪侧不唯一
#         try:
#             _ = llm_p.merge(bert_p, on=keys, how="left", validate="one_to_many")
#             print("→ 左(LLM) 唯一，右(BERT) 不唯一")
#         except Exception:
#             pass
#         try:
#             _ = llm_p.merge(bert_p, on=keys, how="left", validate="many_to_one")
#             print("→ 右(BERT) 唯一，左(LLM) 不唯一")
#         except Exception:
#             pass
#         # 无校验构造中间表
#         tmp = llm_p.merge(bert_p, on=keys, how="left")
#     # 再探测 (LLM×BERT) × ETHER
#     try:
#         merged = tmp.merge(eth_p, on=keys, how="left", validate="one_to_one")
#         print("(LLM×BERT)×ETHER ✅ one_to_one")
#     except Exception as e:
#         print("(LLM×BERT)×ETHER ❌", e)
#         # 判定哪侧不唯一
#         try:
#             _ = tmp.merge(eth_p, on=keys, how="left", validate="one_to_many")
#             print("→ 中间表 唯一，ETHER 不唯一")
#         except Exception:
#             pass
#         try:
#             _ = tmp.merge(eth_p, on=keys, how="left", validate="many_to_one")
#             print("→ ETHER 唯一，中间表 不唯一")
#         except Exception:
#             pass
#         if FORCE_UNIQUE_FALLBACK:
#             print("⚠️ 启用兜底：强制每侧按键唯一（保留首条），以跑通合并。")
#             llm_pf  = force_unique_by_keys(llm_p,  keys)
#             bert_pf = force_unique_by_keys(bert_p, keys)
#             eth_pf  = force_unique_by_keys(eth_p,  keys)
#             merged = (llm_pf
#                       .merge(bert_pf, on=keys, how="left", validate="one_to_one", sort=False)
#                       .merge(eth_pf,  on=keys, how="left", validate="one_to_one", sort=False))
#         else:
#             raise
#     print("LLM行数(聚合后)：", len(f1u), " 合并结果行数：", len(merged))
#     return merged, f0u, f1u, f2u


# # ===================== 调用示例 =====================
# # merged, f0u, f1u, f2u = run_merge_pipeline(f0, f1, f2, keys)
# # display(merged.head())\
# keys =['sentence', 'label_gold', 'gold_start', 'gold_end', 'gold_text']
# keys1=['gold_start','gold_end','sentence','label_gold']
# merged, f0u, f1u, f2u = run_merge_pipeline(df_align_sorted, df_align1_sorted, df_align2_sorted, keys1)



In [ ]:
###used now

# import pandas as pd

# def merge_three_simple(f0, f1, f2, keys):
#     # 0) 列存在性检查
#     for i, df in enumerate([f0, f1, f2], start=0):
#         missing = [k for k in keys if k not in df.columns]
#         if missing:
#             raise KeyError(f"f{i} 缺少这些列: {missing}")
#     # 1) 统一 keys 中字符串列的空白（可选但推荐，避免空格差异）
#     def _normalize(df):
#         df = df.copy()
#         for k in keys:
#             if pd.api.types.is_string_dtype(df[k]):
#                 df[k] = df[k].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
#         return df
#     f0 = _normalize(f0); f1 = _normalize(f1); f2 = _normalize(f2)
#     # 2) 排序 + 去重
#     def _prep(df):
#         return (df.sort_values(by=keys, kind="mergesort")
#                   .drop_duplicates(subset=keys, keep="first")
#                   .reset_index(drop=True))
#     f0u, f1u, f2u = _prep(f0), _prep(f1), _prep(f2)
#     # 3) 三表交集对齐
#     k0 = set(map(tuple, f0u[keys].to_numpy()))
#     k1 = set(map(tuple, f1u[keys].to_numpy()))
#     k2 = set(map(tuple, f2u[keys].to_numpy()))
#     common = k0 & k1 & k2
#     f0u = f0u[f0u[keys].apply(tuple, axis=1).isin(common)]
#     f1u = f1u[f1u[keys].apply(tuple, axis=1).isin(common)]
#     f2u = f2u[f2u[keys].apply(tuple, axis=1).isin(common)]
#     # 4) 前缀 + 合并（以 f1 为基表）
#     def _pref(df, prefix):
#         return df.rename(columns={c: f"{prefix}{c}" for c in df.columns if c not in keys})
#     llm_p  = _pref(f1u, "llm_")
#     bert_p = _pref(f0u, "bert_")
#     eth_p  = _pref(f2u, "ether_")
#     try:
#         merged = (llm_p.merge(bert_p, on=keys, how="left", validate="one_to_one", sort=False)
#                          .merge(eth_p,  on=keys, how="left", validate="one_to_one", sort=False))
#     except Exception as e:
#         # 兜底：强制每侧唯一再合并；若还失败，抛出详细错误
#         def _force(df):
#             return (df.sort_values(by=keys, kind="mergesort")
#                       .drop_duplicates(subset=keys, keep="first"))
#         llm_pf, bert_pf, eth_pf = _force(llm_p), _force(bert_p), _force(eth_p)
#         try:
#             merged = (llm_pf.merge(bert_pf, on=keys, how="left", validate="one_to_one", sort=False)
#                              .merge(eth_pf,  on=keys, how="left", validate="one_to_one", sort=False))
#         except Exception as e2:
#             raise RuntimeError(f"合并失败。可能是键值类型不一致或数据仍非唯一。\n原始异常1: {e}\n兜底异常2: {e2}")
#     return merged, f0u, f1u, f2u


f0 = df_align_sorted
f1 = df_align1_sorted
f2 = df_align2_sorted
# keys = ['gold_start','gold_end','sentence','label_gold']

# merged, f0u, f1u, f2u = merge_three_simple(f0, f1, f2, keys)
# print(len(f1u), len(merged))  # merged 行数应与 f1u 相同

# 假设 f0, f1, f2 的行数和 keys 顺序完全一致
merged = pd.concat([f0.reset_index(drop=True).add_prefix("bert_"),
                    f1.reset_index(drop=True).add_prefix("llm_"),
                    f2.reset_index(drop=True).add_prefix("eth_")],
                   axis=1)



In [ ]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd

# =========== 配置 ===========
# 需要在 merged 里存在的列：
COLS_REQUIRED = [
    "bert_gold_start","llm_gold_start","eth_gold_start",
    "bert_gold_end","llm_gold_end","eth_gold_end",
    "bert_label_gold","llm_label_gold","eth_label_gold",
    "bert_sentence","eth_sentence","llm_sentence"
]
# 不一致行号最多打印多少个
SHOW_N = 50

# =========== 工具函数 ===========
def ensure_cols(df: pd.DataFrame, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"merged 缺少这些列：{missing}")

def nan_equalized(series: pd.Series) -> np.ndarray:
    """把 NaN 统一成占位 '<NA>'，便于相等比较；返回 np.array"""
    return series.astype(object).where(series.notna(), "<NA>").to_numpy()

def compare_triplet_eq(df: pd.DataFrame, a: str, b: str, c: str, name: str, show_n=SHOW_N):
    """逐行比较三列是否相等（NaN 视为相等），打印统计与不一致行号。"""
    x, y, z = nan_equalized(df[a]), nan_equalized(df[b]), nan_equalized(df[c])
    same = (x == y) & (x == z)
    total = len(df)
    print(f"[{name}] 三方完全一致: {int(same.sum())}/{total}")
    if not same.all():
        idx_xy = np.where(x != y)[0].tolist()
        idx_xz = np.where(x != z)[0].tolist()
        idx_yz = np.where(y != z)[0].tolist()
        bad_all = np.where(~same)[0].tolist()
        print(f"  - 任意两者不等的行号(最多{show_n}): {bad_all[:show_n]}")
        print(f"    · {a} vs {b} 不同: {idx_xy[:show_n//3]}")
        print(f"    · {a} vs {c} 不同: {idx_xz[:show_n//3]}")
        print(f"    · {b} vs {c} 不同: {idx_yz[:show_n//3]}")
    return same

def normalize_spaces(s: pd.Series) -> pd.Series:
    """折叠连续空白为单空格，并去首尾空白。"""
    return s.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

def compare_pair_eq(df: pd.DataFrame, a: str, b: str, name: str, ignore_spaces=False, show_n=SHOW_N):
    """两列逐行比较是否相等；可选择忽略空白差异；打印统计与不一致行号。"""
    s1 = df[a]
    s2 = df[b]
    if ignore_spaces:
        s1 = normalize_spaces(s1)
        s2 = normalize_spaces(s2)
        tag = f"{name}（忽略空白差异）"
    else:
        tag = f"{name}（原样比较）"
    x, y = nan_equalized(s1), nan_equalized(s2)
    same = (x == y)
    total = len(df)
    print(f"[{tag}] 完全一致: {int(same.sum())}/{total}")
    if not same.all():
        bad = np.where(~same)[0].tolist()
        print(f"  - 不一致的行号(最多{show_n}): {bad[:show_n]}")
    return same

# =========== 主检查 ===========
def run_all_checks(merged: pd.DataFrame):
    ensure_cols(merged, COLS_REQUIRED)
    print("==== 关键字段三方一致性（NaN 视为相等）====")
    compare_triplet_eq(merged, "bert_gold_start", "llm_gold_start", "eth_gold_start", "gold_start")
    compare_triplet_eq(merged, "bert_gold_end",   "llm_gold_end",   "eth_gold_end",   "gold_end")
    compare_triplet_eq(merged, "bert_label_gold", "llm_label_gold", "eth_label_gold", "label_gold")
    print("\n==== 句子一致性（原样对比）====")
    compare_pair_eq(merged, "bert_sentence", "eth_sentence", "bert_sentence vs eth_sentence", ignore_spaces=False)
    compare_pair_eq(merged, "bert_sentence", "llm_sentence",      "bert_sentence vs llm_sentence",      ignore_spaces=False)
    compare_pair_eq(merged, "eth_sentence",  "llm_sentence",      "eth_sentence  vs llm_sentence",      ignore_spaces=False)
    print("\n==== 句子一致性（忽略空白差异）====")
    compare_pair_eq(merged, "bert_sentence", "eth_sentence", "bert_sentence vs eth_sentence", ignore_spaces=True)
    compare_pair_eq(merged, "bert_sentence", "llm_sentence",      "bert_sentence vs llm_sentence",      ignore_spaces=True)
    compare_pair_eq(merged, "eth_sentence",  "llm_sentence",      "eth_sentence  vs llm_sentence",      ignore_spaces=True)

# =========== 运行 ===========
# 直接调用：
run_all_checks(merged)


In [ ]:
# import pandas as pd
# from functools import reduce
# df_align1["sentence"] = df_align1["sentence_gold"].fillna(df_align1["sentence_pred"])
# df_align2["sentence"] = df_align2["sentence_gold"].fillna(df_align2["sentence_pred"])
# # 你的三张表
# # df_align1 -> LLM
# # df_align  -> BERT
# # df_align2 -> ETHER

# key_cols = ["sentence","label_gold","gold_start","gold_end",
#             "gold_text","label_pred","pred_start","pred_end"]

# # 1) 每张表先在 key 上去重（防止一对多）
# def dedup_on_keys(df):
#     return df.drop_duplicates(subset=keys1, keep="first").copy()
#     # return df.copy()


# llm  = dedup_on_keys(df_align1_sorted)
# bert = dedup_on_keys(df_align_sorted)
# ether= dedup_on_keys(df_align2_sorted)

# def add_prefix_keep_keys(df, prefix):
#     out = df.add_prefix(prefix)                 # 全加前缀
#     for c in key_cols:                          # 再把 key 恢复成原名（供 join）
#         out[c] = df[c]
#     return out

# llm_p   = add_prefix_keep_keys(llm,   "llm_")
# bert_p  = add_prefix_keep_keys(bert,  "bert_")
# ether_p = add_prefix_keep_keys(ether, "ether_")

# # 3) 三路外连接（按同一组 key），逐步 merge
# dfs = [llm_p, bert_p, ether_p]
# merged_all = reduce(
#     lambda left, right: pd.merge(left, right, on=key_cols, how="outer", validate="one_to_one"),
#     dfs
# )
# print("merged_all shape:", merged_all.shape)
# print(merged_all.head(2))

# # 可选：合并后按“实体唯一键”去重再做统计，避免行膨胀影响计数
# # merged_all.loc[merged_all["bert_match_type"]=="M", key_cols].drop_duplicates().shape[0]
# merged_all['bert_match_type'].value_counts(dropna=False)

In [ ]:
cols_to_keep = [
    "bert_match_type",
    "llm_match_type",
    "eth_match_type",
    "bert_label_gold",
    "bert_gold_start",
    "bert_gold_end",
    "bert_gold_text",
    "bert_sentence"
]

merged_sub = merged[cols_to_keep]

print(merged_sub.head())
merged_sub.to_csv("merged.csv", index=False)

In [ ]:
merged.to_csv("merged_all.csv", index=False)

In [ ]:
# import pandas as pd
# from functools import reduce

# # 你已有的键
# key_cols = ["sentence","label_gold","gold_start","gold_end",
#             "gold_text","label_pred","pred_start","pred_end"]

# def report_dupes(df: pd.DataFrame, name: str, keys):
#     """报告 df 在 keys 上的非唯一键组合"""
#     g = df.groupby(keys).size().reset_index(name="count")
#     dup = g[g["count"] > 1].sort_values("count", ascending=False)
#     print(f"[{name}] 非唯一键组合数: {len(dup)}")
#     if len(dup):
#         print(dup.head(10))  # 只展示前 10 组
#     return dup

# def check_one_to_one(left: pd.DataFrame, right: pd.DataFrame, keys, lname="LEFT", rname="RIGHT"):
#     """合并前检查左右在 keys 上是否 one-to-one"""
#     print(f"\n=== 检查 {lname} 与 {rname} 在 keys 上是否唯一 ===")
#     du_l = report_dupes(left,  lname, keys)
#     du_r = report_dupes(right, rname, keys)
#     return du_l, du_r

# def debug_merge(left: pd.DataFrame, right: pd.DataFrame, keys, how="outer",
#                 lname="LEFT", rname="RIGHT"):
#     """带体检与错误定位的 merge（validate=one_to_one）"""
#     # 先检查唯一性
#     du_l, du_r = check_one_to_one(left, right, keys, lname, rname)
#     try:
#         out = pd.merge(left, right, on=keys, how=how, validate="one_to_one")
#         print(f"[OK] 合并成功：{lname} x {rname} -> out.shape={out.shape}")
#         # 合并后再检查一次（理论上应为一对一）
#         _ = report_dupes(out, f"{lname}x{rname}", keys)
#         return out
#     except pd.errors.MergeError as e:
#         print(f"[ERROR] 合并失败：{lname} x {rname}")
#         print("原因：", e)
#         # 进一步定位：哪些键同时在左右都出现重复，导致多对多
#         print("\n>>> 进一步定位：左右同时重复的键组（可能导致多对多）")
#         gL = left.groupby(keys).size().reset_index(name="cntL")
#         gR = right.groupby(keys).size().reset_index(name="cntR")
#         both = pd.merge(gL[gL["cntL"] > 1], gR[gR["cntR"] > 1], on=keys, how="inner")
#         if len(both) == 0:
#             print("没有发现两侧同时重复；很可能是一侧重复导致 one_to_one 不满足。")
#             print("左侧前 10 个重复键组：")
#             print(gL[gL["cntL"] > 1].sort_values("cntL", ascending=False).head(10))
#             print("右侧前 10 个重复键组：")
#             print(gR[gR["cntR"] > 1].sort_values("cntR", ascending=False).head(10))
#         else:
#             print(both.sort_values(["cntL","cntR"], ascending=False).head(20))
#         # 返回 None 方便你中断流程
#         return None

# # ---- 你的三张已经加过前缀并“还原了 key 列”的 DataFrame ----
# # llm_p, bert_p, ether_p

# # 逐步合并并调试
# merged12 = debug_merge(llm_p, bert_p, key_cols, how="outer", lname="LLM", rname="BERT")
# if merged12 is None:
#     raise SystemExit("先修复 LLM 和 BERT 的重复问题，再继续。")

# merged_all = debug_merge(merged12, ether_p, key_cols, how="outer", lname="(LLM×BERT)", rname="ETHER")
# if merged_all is None:
#     raise SystemExit("先修复 ETHER 与(LLM×BERT) 的重复问题。")

# print("最终 merged_all.shape =", merged_all.shape)


In [ ]:
# # 只看 label_gold、gold_start、gold_end、sentence 是否完全重复
# dupes = df_m[df_m.duplicated(subset=["case_id","label_gold", "gold_start", "gold_end"], keep=False)]

# print(f"重复行数: {len(dupes)}")
# print(dupes)

# unique_count = df_m[["label_gold", "gold_start", "gold_end", "sentence"]].drop_duplicates().shape[0]
# print(f"唯一组合数量: {unique_count}")


In [19]:
# import spacy
# from spacy.training import Corpus
# from spacy.training.example import Example
# from collections import defaultdict

# # Load your trained model
# nlp = spacy.load("model-best")

# # Load validation data
# test_corpus = Corpus("Dataset_spacy/valid_FAERS_R1_v1_SME1_new_revision.spacy")
# test_data = list(test_corpus(nlp))

In [22]:
# # Store detailed predictions
# detailed_results = []
# total_metrics = []
# for i, example in enumerate(test_data, 1):
#     # Run the model
#     pred_doc = nlp(example.text)
#     # Gold standard (true) entities
#     gold_entities = [(ent.start_char, ent.end_char, ent.label_) for ent in example.reference.ents]
#     # Predicted entities
#     pred_entities = [(ent.start_char, ent.end_char, ent.label_) for ent in pred_doc.ents]
#     eval_metrics = eval_pred(gold_entities, pred_entities, i)
#     # Append result for this sample
#     detailed_results.append({
#         "text": example.text,
#         "gold_entities": gold_entities,
#         "pred_entities": pred_entities
#     })
    
#     total_metrics += eval_metrics




In [ ]:
# import pandas as pd

# df_bert= pd.DataFrame(total_metrics)

In [ ]:
# cols = [0]+[1] + list(range(df_bert.shape[1] - 5, df_bert.shape[1]))
# df_bert.sort_values(by="case_id").iloc[:, cols]

In [ ]:
# df_bert

In [ ]:
# df_bert.groupby('label')[['precision', 'recall','f1']].mean()

In [ ]:
# detailed_results[3]